# **Recurrent Neural Networks**

## Table of Contents
- [Theory](#theory)
- [Text Classification](#app-1)
- [Next Frame Prediction](#app-2)

## **Theory** <a class="anchor" id="theory"></a>

### **Generalities** - why previous models don't work on sequential data?

Given a sequence of $T \geq 2$ temporal states $x_0, x_1, \dots, x_{T-1}$, with $x_i \in \mathbb{R}^d$ for $i \in \{0, \dots, T-1\}$, we want to extract task-relevant information while preserving the temporal structure of the sequence.

Standard MLPs are inherently order-agnostic and cannot capture temporal dependencies. CNNs offer a step forward — by sliding a kernel along the temporal dimension, they can capture local patterns. However, since the **ordering of elements matters**, we must apply **causal convolution**, ensuring that the output at each timestep depends only on the current and past states — never on future ones.

In a standard CNN, the output $y_i$ depends on a **symmetric context window** centered at $x_i$:

$$[x_{i - \lfloor K/2 \rfloor},\ x_{i - \lfloor K/2 \rfloor + 1},\ \dots,\ x_i,\ \dots,\ x_{i + \lfloor K/2 \rfloor - 1},\ x_{i + \lfloor K/2 \rfloor}]$$

In contrast, **causal convolution** restricts the context to only past and present states:

$$[x_{i - K + 1},\ x_{i - K + 2},\ \dots,\ x_{i-1},\ x_i]$$

where $K \geq 3$ is the kernel size. Note that for standard CNNs, $K$ is additionally required to be odd (i.e., $K \bmod 2 = 1$) to allow symmetric padding, whereas causal convolutions work with any kernel size $K \geq 1$.

Long-term dependencies that may arise in the sequence $\{x_i\}_{i=1}^T$ would be lost due to the limiting context length $K$. On the other hand, increasing $K$ leads to a higher memory and computational cost, along with the possibility of bypassing short-term information.

### **RNNs**

**Recurrent Neural Networks (RNNs)** are a class of NNs tailored to temporal/sequential data, step-by-step processing each input $x_t$ using the same set of parameters, taking into account the *context* of previous steps.

<div style="text-align: center;">
  <img src="imgs/rnn.png" alt="description" width="1000"/>
  <figcaption>Diagram of a 1-layer RNN</figcaption>
</div>

The **same RNN cell** is reused at every timestep — receiving the current input $x_i$ and the previous hidden state $h_{i-1}$ (along with any additional feature vectors) — and producing the updated hidden state $h_i$. This weight sharing across time is what distinguishes RNNs from feed-forward models.

As illustrated above, the left diagram shows the **rolled** (compact) view of a single RNN cell, while the right shows it **unrolled** across $T$ timesteps, making the sequential data flow explicit. Multiple such cells can be stacked to form a **multi-layer RNN**, where each upper layer takes as input the hidden states $h_i$ produced by the layer below.

Despite their appeal, classic RNNs come with notable limitations:

- **Vanishing long-term dependencies** — the hidden state $h_i$ acts as a bottleneck, compressing the entire history into a fixed-size vector. Relevant information from distant past timesteps tends to fade, making it difficult for the network to capture long-range dependencies.
- **Lack of parallelism** — due to the sequential nature of the recurrence ($h_i$ depends on $h_{i-1}$), each timestep must wait for the previous one to complete. This makes RNNs inherently slow to train on long sequences and difficult to scale.

## The **LSTM** cell

[Long Short-Term Memory](https://www.bioinf.jku.at/publications/older/2604.pdf) (LSTM) cells were introduced to address the vanishing long-term dependency problem inherent in classic RNNs.

Their key innovation is the **cell state** $c_i$ — an additional recurrent vector that flows through all timesteps with only small, controlled modifications at each step. Unlike the hidden state $h_i$, which is recomputed entirely at every timestep, the cell state acts as a **long-term memory lane**: it accumulates and preserves information over extended time horizons, selectively retaining or discarding content through a set of learned gating mechanisms.

Each LSTM cell consists of four interconnected computational sub-modules:

1. **Forget Gate** — takes $\{h_{t-1}, x_t\}$ as input and produces a mask $f_t \in [0, 1]^d$, controlling **what fraction of the previous cell state** $C_{t-1}$ to retain. Values close to 0 cause forgetting; values close to 1 preserve information.

2. **Input Gate** — takes $\{h_{t-1}, x_t\}$ and produces a mask $i_t \in [0, 1]^d$ alongside a candidate cell state $\tilde{C}_t$. The mask $i_t$ controls **how much of the new candidate** $\tilde{C}_t$ is incorporated into the cell state.

3. **Cell State Update** — combines the forget and input gates to update the cell state:
$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$
where $\odot$ denotes element-wise multiplication. This additive update is what allows gradients to flow across long time horizons.

4. **Output Gate** — takes $\{h_{t-1}, x_t\}$ and produces a mask $o_t \in [0, 1]^d$, determining **which elements of the cell state** are exposed as the hidden state $h_t = o_t \odot \tanh(C_t)$ at the current timestep.

<div style="text-align: center;">
  <img src="imgs/lstm.png" alt="description" width="1000"/>
  <figcaption>Information flow in a LSTM cell, for a single timestep.</figcaption>
</div>

### Further Reading — LSTM Variants

- [Bidirectional LSTM](https://en.wikipedia.org/wiki/Bidirectional_recurrent_neural_networks) — processes the sequence in both forward and backward directions, enriching the hidden state with future context as well.
- [ConvLSTM](https://proceedings.neurips.cc/paper/2015/hash/07563a3fe3bbe7e3ba84431ad9d055af-Abstract.html) — replaces the fully connected operations in the LSTM gates with convolutions, making it suitable for spatiotemporal data.
- [Self-Attention LSTM](https://ojs.aaai.org/index.php/AAAI/article/view/6819) — augments the LSTM with a self-attention mechanism, allowing the model to directly attend to relevant past hidden states.

---

## *{Self, Cross}* - **Attention**

Although not strictly tied to recurrent architectures, **attention mechanisms** have been widely adopted across many domains as a powerful tool for capturing dependencies — by computing similarity scores between different parts of the input, a model can dynamically focus on the most relevant regions of the data.

Neural machine translation (NMT) was among the first sequential tasks to benefit from [attention mechanisms](https://arxiv.org/pdf/1409.0473). Classical encoder-decoder architectures compressed the entire source sentence into a single fixed-length vector — a significant bottleneck, especially for long sequences. Attention alleviates this by allowing the decoder to **softly search** over all encoder hidden states at each decoding step, assigning higher weight to the source positions most relevant to predicting the current target word.

The **multi-head self-attention** works as follows:

1. For each head, the input sequential data $X\in\mathbb{R}^{T \times D}$, with $T$ being the temporal dimension (or any multide of dimensions on which we can define some "ordering"), is linearly transformed into **query / key / value** tensors:
$$Q_h \gets X W_h^{Q}, \quad K_h \gets X W_h^{K}, \quad V_h \gets X W_h^{V}$$ 
$$W_h^{Q}, W_h^{K}, W_h^{V} \in \mathbb{R}^{D \times D'}$$
$$Q, K, V \in \mathbb{R}^{T\times D'}$$
$$h \in \{1, \dots, n_{heads}\}$$

2. Similarity scores are computed between keys and queries *Softmax*:
$$A_h \gets \text{\texttt{softmax}}(Q_h K_h^T), \quad A_h \in [0, 1]^{T\times T}$$

3. These scores are used to mix the temporal entries of $V_h$, computing at each timestep a convex combination of the previous values:
$$Z_h \gets A_h V_h, \quad Z_h \in \mathbb{R}^{T\times D}$$

4. All outputs $Z_h$ from each head $h$ are concatenated and multiplied by a final output matrix $W_o \in \mathbb{R}^{D' \cdot n_{heads} \times D}$, resulting the final features $Z$:
$$Z \gets \big[Z_1 \Vert Z_2 \Vert \dots \Vert Z_{n_{heads}}\big] W_o, \quad Z \in \mathbb{R}^{T \times D}$$

<div style="text-align: center;">
  <img src="imgs/self_att.png" alt="description" width="1000"/>
  <figcaption><b>Left: </b>Computing Q, K, V. <b>Right: </b>Multi-head self-attention. Sub-images modified from <a href="https://jalammar.github.io/illustrated-transformer/">here</a></figcaption>
</div>

**(Multi-Head) Cross-Attention** extends the self-attention formulation to the case where **two distinct sequences** serve as input — a query sequence and a key/value sequence, which need not have the same temporal length $T$. Attention scores $A_h$ are then computed *across* the two sources, allowing each position in one sequence to attend to relevant positions in the other.

Cross-Attention has found particularly strong adoption in **multi-modal learning**, where it computes similarities between encoded representations of different modalities to retrieve relevant, aligned responses. Most notably, [Contrastive Language-Image Pretraining (CLIP)](https://arxiv.org/abs/2103.00020) aligns text and image embeddings in a shared latent space, enabling zero-shot classification without task-specific training. Another example is [CrossViT](https://openaccess.thecvf.com/content/ICCV2021/papers/Chen_CrossViT_Cross-Attention_Multi-Scale_Vision_Transformer_for_Image_Classification_ICCV_2021_paper.pdf), which applies cross-attention between features extracted by two parallel transformer encoders at different scales, fusing them into a richer multi-scale representation.

---

## **Movie review classification** <a class="anchor" id="app-1"></a>

<div style="text-align: center;">
  <img src="imgs/sst2_sentiment_classification.svg" alt="description" width="800"/>
</div>

## Loading Data

We'll use `torchtext` package to download and prepare our data. 

In [29]:
import torch
torch.__version__

'2.0.1'

Install `torchtext` without overriding the current `torch` install. Take at look at [this table](https://pypi.org/project/torchtext/) and [these releases](https://github.com/pytorch/text/releases) to make sure you install the right `torchtext` version for your current `torch` installation. You'll also need `torchdata` - check out [its releases](https://github.com/pytorch/data/releases) to see which version you need. `poratlocker==2.8.2` is required, according to [this fix](https://github.com/pytorch/text/issues/2172#issuecomment-1808401332).

In [30]:
!pip install torchtext==0.15.2 torchdata==0.6.1 
!pip install portalocker==2.8.2

Load the **SST-2** (Stanford Sentiment Treebank) dataset, consisting of sentence and binary sentiment label pairs — where $0$ denotes *negative* and $1$ denotes *positive* sentiment.

In [31]:
import torch
import torchtext
from torchtext.datasets import SST2

train_ds = SST2(root="data/", split="train")
test_ds = SST2(root="data/", split="dev")

label_names = {
    0: "bad",
    1: "good"
}

In [32]:
seen_classes = {}
max_per_class_samples = 3

for sample in iter(test_ds):
    x, y = sample

    if y not in seen_classes.keys():
        print(f"Class y={y} ({label_names[y]}): {x}")
        seen_classes[y] = 1
    else:
        if seen_classes[y] < max_per_class_samples:
            print(f"Class y={y} ({label_names[y]}): {x}")
            seen_classes[y] += 1

Class y=1 (good): it 's a charming and often affecting journey .
Class y=0 (bad): unflinchingly bleak and desperate
Class y=1 (good): allows us to hope that nolan is poised to embark a major career as a commercial yet inventive filmmaker .
Class y=1 (good): the acting , costumes , music , cinematography and sound are all astounding given the production 's austere locales .
Class y=0 (bad): it 's slow -- very , very slow .
Class y=0 (bad): a sometimes tedious film .


## Sentence Pre-Processing

We'll pre-process each sequence in the following manner:
$$\text{Sentence} \xrightarrow{\text{Tokenizer}} [\text{Words}] \xrightarrow{Vocabulary} [\text{Word Indexes}]$$
For that, we'll need a basic english sentence (word) breaker, and a vocabulary to assigning an integer index to each unique word in our corpus.

In [33]:
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

tokenizer = get_tokenizer("basic_english")

def yield_tokens(ds):
    for text, y in ds:
        yield tokenizer(text)

vocab = build_vocab_from_iterator(yield_tokens(train_ds), specials=["<unk>"])
vocab.set_default_index(vocab["<unk>"])

In [34]:
x, y = next(iter(test_ds))

print(f"label-sentence: {label_names[y]} \ {x}")
print("Tokenized sequence: ", tokenizer(x))
print("Vocabulary indices for the tokenized sequence: ", vocab(tokenizer(x)))

label-sentence: good \ it 's a charming and often affecting journey .
Tokenized sequence:  ['it', "'", 's', 'a', 'charming', 'and', 'often', 'affecting', 'journey', '.']
Vocabulary indices for the tokenized sequence:  [13, 7, 9, 3, 296, 4, 144, 1411, 596, 6]


Since we cannot directly stack multiple tokenized sequences into a `(batch_size, n_words)` tensor of indices, we'll need to pad each sentence with `<unk>` characters until the largest n.o. words in a batch is reached.

We'll do this by passing a custom `collate_fn` to the dataloaders, which will automatically perform this operation on each extracted batch of sentences.

In [35]:
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}")

# This will be applied over each sentence -> just like the above example
text_processor = lambda x: vocab(tokenizer(x)) 
label_processor = lambda y: int(y)

# This will be applied over each extracted batch of sentences
def collate_fn(batch):
    label_list, text_list, length_list = [], [], []
    for _text, _label in batch:
        label_list.append(label_processor(_label))
        processed_text = torch.LongTensor(text_processor(_text))
        text_list.append(processed_text)
        length_list.append(len(processed_text))
    
    label_list = torch.tensor(label_list, dtype=torch.int64)
    text_list = torch.nn.utils.rnn.pad_sequence(text_list, batch_first=True, padding_value=vocab["<unk>"]) # returns a (B, T_max) tensor of indices
    length_list = torch.LongTensor(length_list)

    # need to sort in descending order - s.t. the first element has the highest length in the batch
    # this is useful for what will follow in the training phase
    length_list, perm_idx = length_list.sort(0, descending=True)
    text_list = text_list[perm_idx]
    label_list = label_list[perm_idx]
    
    return text_list.to(device), label_list.to(device), length_list.to(device)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

Using cpu


In [36]:
for batch in train_loader:
    x, y, lengths = batch

    bs, max_size = x.shape[0], x.shape[1]

    print("Batch size: ", bs)
    print("Max size in batch: ", max_size)
    print("Batch tensor shape: ", x.shape)
    print("All lenghts in batch: ", lengths)

    print("First 3 examples in batch:")
    for i in range(3):
        print("-> ", x[i].cpu().numpy())
    
    break

Batch size:  64
Max size in batch:  44
Batch tensor shape:  torch.Size([64, 44])
All lenghts in batch:  tensor([44, 28, 23, 22, 19, 19, 19, 18, 17, 17, 17, 16, 16, 16, 16, 15, 14, 14,
        12, 12, 12, 11, 11, 11, 10, 10,  9,  9,  8,  8,  8,  8,  7,  7,  6,  6,
         6,  6,  6,  6,  5,  5,  5,  5,  5,  4,  4,  4,  4,  3,  3,  3,  3,  3,
         3,  3,  2,  2,  2,  2,  2,  1,  1,  1])
First 3 examples in batch:
->  [   6    6    6    3 1524   75   20   32   18  801   16 1258    5    1
 2726   11 8921    1 3958 7519    5    1 8993    7    9    6    6    6
    1   17    7    9  428   34    3   73   64  100   13   43   19  355
    7    7]
->  [  11    7    9  283   86  148   22  163   13    2 1293  366  141  177
    4 2756 1648   48  383 4130    6    3    6    7    9 8790    4 4964
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0]
->  [2102   10 8291   28 7229 2033    7    9 4141 3879    2  114 3074    2
    4  151    5   11    2   37   17 6700    6  

A `nn.Embedding` layer is used to convert these integer indexes into float tensor representations. An embedding layer works as a lookup table, mapping word indexes to real-valued tensors, which are optimized during training along with all the other parameters.

<div style="text-align: center;">
  <img src="imgs/embedd.png" alt="description" width="800"/>
  <figcaption>Simplified diagram of sentence embedding.</figcaption>
</div>

In [37]:
import torch.nn as nn

example_embedd = nn.Embedding(len(vocab), 50)
embedded_x = example_embedd(x.cpu()) # computes the embedding separately and independently for each word (index)
bs, seq, emb = embedded_x.shape
print(f"Batch \ Seq.Len \ Embedding dim: {bs} \ {seq} \ {emb}")

Batch \ Seq.Len \ Embedding dim: 64 \ 44 \ 50


## Build and Train Embeddings + LSTM

In order to efficiently work with batches containing variable-length data (effective length) we'll need to `pad-pack` each batch s.t. a batch is presented to the LSTM as follows:

<div style="text-align: center;">
  <img src="imgs/pack_padded.png" alt="description" width="1200"/>
  <figcaption>Pack-padding strategy for training on variable-sized sentences.</figcaption>
</div>

In [38]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedd_size, num_layers, lstm_hidden_dim, n_classes, 
                 bidirectional=False, predict_on_output=True):
        super(RNNClassifier, self).__init__()

        self.embedd_size = embedd_size
        self.embedding = nn.Embedding(vocab_size, embedd_size, 
                                     padding_idx=0, 
                                     scale_grad_by_freq=False,
                                     max_norm=1.0) # this receives indices and returns float tensor representations

        self.bidirectional = bidirectional
        self.lstm = nn.LSTM(embedd_size, lstm_hidden_dim, 
                            dropout=0.1, 
                            num_layers=num_layers, 
                            batch_first=True, 
                            bidirectional=bidirectional)

        # We can choose between predicting on the output tensor, or on the hidden state
        if predict_on_output:
            if bidirectional:
                # In the bidirectional case, we'll have two two outputs - one for ->, another for <-
                self.out_dim = lstm_hidden_dim * 2
            else:
                self.out_dim = lstm_hidden_dim
        else:
            # predict on the concatenation of last hidden states, from each layer
            if bidirectional:
                self.out_dim = lstm_hidden_dim * num_layers * 2
            else:
                self.out_dim = lstm_hidden_dim * num_layers

        self.out_classification = nn.Sequential(
            nn.Linear(self.out_dim, n_classes) 
        )
        
        self.predict_on_output = predict_on_output
        self.num_layers = num_layers
    
    def forward(self, x, lengths):
        embeddings = self.embedding(x)
        
        # need to pack-padd embeddings, in order to exclude padded values from computation in LSTM
        embeddings_ = torch.nn.utils.rnn.pack_padded_sequence(embeddings, lengths.cpu().numpy(), batch_first=True).to(device)

        # hidden and cell states will be returned for each LSTM layer - (num_layers, batch_size, lstm_hidden_dim)
        out, (hidden_state, cell_state) = self.lstm(embeddings_)

        # unpack padded output sequence
        out_, lens_unpacked  = torch.nn.utils.rnn.pad_packed_sequence(out, batch_first=True)

        if self.predict_on_output:
            # take the output for each element in the batch, at the index of len_sequence - 1 (last word)
            preds = self.out_classification(
                        torch.gather(
                            out_, dim=1, 
                            index=(lens_unpacked - 1).view(-1, 1).unsqueeze(2).expand(-1, -1, out_.size(-1)).to(out_.device)
                        ).squeeze()
                    )
        else:
            # Classify on the concatenation of hidden_states from each layer
            preds = self.out_classification(
                        torch.cat([hidden_state[i] for i in range(self.num_layers)], dim=1)
                    )  
        
        return preds

Define hyperparameters:

In [39]:
embedd_size = 50
num_layers = 2
lstm_hidden_dim = 32
bidirectional = False
predict_on_output = True
n_classes = 2

epochs = 5
lr = 1e-3
l2 = 0

In [40]:
model = RNNClassifier(len(vocab), 
                      embedd_size, 
                      num_layers, 
                      lstm_hidden_dim, 
                      n_classes,
                      bidirectional=bidirectional,
                      predict_on_output=predict_on_output).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=l2)

folder_path = "models/sst2/"
file_name = "model.pth"

print("Model output shape: ", model(x, lengths).shape)

Model output shape:  torch.Size([64, 2])


After training, you'll observe some overfitting - you can play with $\ell_2$-regularization, but be careful not to assign a too big weighting factor, otherwise it will not learn at all.

In [41]:
import importlib
import torch_utils
importlib.reload(torch_utils)

train_losses, test_losses = torch_utils.train_loop(
    model, train_loader, optimizer, criterion, epochs, 
    test_loader=test_loader, 
    device=device, 
    folder_path=folder_path, 
    file_name=file_name
)

1053it [00:17, 60.08it/s]
14it [00:00, 112.05it/s]


Epoch 1/5: train_loss=0.16924819973247004 train_acc=0.9378906888001307 test_loss=0.5718128042561668 test_acc=0.8245412844036697


1053it [00:16, 62.69it/s]
14it [00:00, 119.19it/s]


Epoch 2/5: train_loss=0.14693281258273327 train_acc=0.9449435032442947 test_loss=0.5101505517959595 test_acc=0.8188073394495413


1053it [00:17, 59.62it/s]
14it [00:00, 114.38it/s]


Epoch 3/5: train_loss=0.12811104087238637 train_acc=0.9512687641984291 test_loss=0.6509220387254443 test_acc=0.8165137614678899


1053it [00:16, 62.05it/s]
14it [00:00, 120.91it/s]


Epoch 4/5: train_loss=0.114315751589771 train_acc=0.9564952709023148 test_loss=0.6681933552026749 test_acc=0.8119266055045872


1053it [00:17, 61.91it/s]
14it [00:00, 116.93it/s]

Epoch 5/5: train_loss=0.10468187930145552 train_acc=0.9595391171361118 test_loss=0.7203564835446221 test_acc=0.8176605504587156


In [42]:
import os
import ipywidgets as widgets
from IPython.display import display, clear_output

model.load_state_dict(
    torch.load(os.path.join(folder_path, file_name))["state_dict"]
)

input_text = widgets.Text(placeholder='Type here...', description='Input:', layout = widgets.Layout(width='1000px'))
output_text = widgets.Output()

def predict_and_update(change):
    with output_text:
        if len(change.new) > 0:
            clear_output()
            sentence = change.new
            length = len(tokenizer(sentence))
            tokens = torch.LongTensor(text_processor(sentence))[None, ...].to(device)
            length = torch.LongTensor([length]).to(device)
            
            print('Prediction:', label_names[model(tokens, length).argmax().item()])

input_text.observe(predict_and_update, names='value')

display(input_text)
display(output_text)

Text(value='', description='Input:', layout=Layout(width='1000px'), placeholder='Type here...')

Output()

### Analysing word embeddings

The `nn.Embedding` layer has optimized a $W \in \mathbb{R}^{|\texttt{Vocab}| \times \texttt{embedd\_size}}$ matrix containing representations for each word in our vocabulary. We can compute distances between the embedding of a word and each entry in the matrix to get the most similar or least similar words.

In [43]:
def get_top_k(k, emb_word, vocab, embeddings, device, most_similar=True, distance="l2", lookup_token=None, return_res=False):
    if distance == "l2":
        distances = torch.norm(embeddings - emb_word, p=2, dim=1)
    elif distance == "cosine":
        distances = 1 - torch.nn.functional.cosine_similarity(emb_word, embeddings, dim=1)
    else:
        raise ValueError(f"Unknown distance {distance}.")
    
    idx_sorted = torch.argsort(distances)

    if most_similar:
        if return_res:
            if lookup_token:
                return [lookup_token(idx_sorted[i + 1]) for i in range(k)]
            else:
                return [vocab.lookup_token(idx_sorted[i + 1]) for i in range(k)]
        else:
            print(f"Top k similar ({distance}): ", end=" ")
            for i in range(k):
                if lookup_token:
                    print(lookup_token(idx_sorted[i + 1]), end=", ")
                else:
                    print(vocab.lookup_token(idx_sorted[i + 1]), end=", ") # if i == 0, it would return that actual word
            print("\n")
    else:
        if return_res:
            if lookup_token:
                return [lookup_token(idx_sorted[-(i + 1)]) for i in range(k)]
            else:
                return [vocab.lookup_token(idx_sorted[-(i + 1)]) for i in range(k)]
        else:
            print(f"Top k disimilar ({distance}): ", end=" ")
            for i in range(k):
                if lookup_token:
                    print(lookup_token(idx_sorted[-(i + 1)]), end=", ") 
                else:
                    print(vocab.lookup_token(idx_sorted[-(i + 1)]), end=", ") # if i == 0, it would return that actual word    
            print("\n")

In [45]:
word = "death"
idx_word = vocab[word]
emb_word = model.embedding(torch.LongTensor([idx_word]).to(device))
k = 5

get_top_k(k, emb_word, vocab, model.embedding.weight, device)
get_top_k(k, emb_word, vocab, model.embedding.weight, device, most_similar=False)
get_top_k(k, emb_word, vocab, model.embedding.weight, device, distance="cosine")
get_top_k(k, emb_word, vocab, model.embedding.weight, device, most_similar=False, distance="cosine")

Top k similar (l2):  nobody, ponderous, beaten, exaggerated, careful, 

Top k disimilar (l2):  werner, reaffirms, conrad, claude, raw-nerved, 

Top k similar (cosine):  perverse, absurdity, generic, stereotypical, manifesto, 

Top k disimilar (cosine):  provides, uncomfortably, sweet, role, recent, 



In [51]:
analogies_to_explore = [
    ("love", "romance", "hate"),
    ("killing", "action", "laughing"),
    ("man", "chef", "woman"),  
    ("man", "doctor", "woman")
]

analogies_students = [
    ("dog", "fur", "cat"),
    ("man", "livingroom", "woman"),
    ("bird", "feather", "elephant")
]

for analogy in analogies_to_explore:
    word_a, word_b, word_c = analogy
    idx_word_a, idx_word_b, idx_word_c = vocab[word_a], vocab[word_b], vocab[word_c]
    
    emb_word_a = model.embedding(torch.LongTensor([idx_word_a]).to(device))
    emb_word_b = model.embedding(torch.LongTensor([idx_word_b]).to(device))
    emb_word_c = model.embedding(torch.LongTensor([idx_word_c]).to(device))

    new_embed = emb_word_b - emb_word_a + emb_word_c
    
    result = get_top_k(3, new_embed, vocab, model.embedding.weight, device, distance="cosine", return_res=True)
    result = list(map(str.upper, result))
    
    if result:
        print(f"{word_a.upper()} is to {word_b.upper()} as {word_c.upper()} is to {result}")
    else:
        print("Some words in the analogy are not in the vocabulary.")

LOVE is to ROMANCE as HATE is to ['SLACK', 'OVERRUN', 'COLLECTION']
KILLING is to ACTION as LAUGHING is to ['FIERCE', 'MEMORABLE', 'INNOVATIVE']
MAN is to CHEF as WOMAN is to ['SUCK', 'BOGS', 'LAISSEZ-PASSER']
MAN is to DOCTOR as WOMAN is to ['ALBEIT', 'INERTIA', 'DISCONTENT']


Things that have affected the quality of the above results:
- Vocabulary was too simple - only words and phrases related to a single subject (i.e. movie reviews)
- The classification task is not too broad - it doesn't help that much in learning meaningful relationships between embeddings 

### Compare with [GloVe](https://nlp.stanford.edu/pubs/glove.pdf) embeddings

[Global Vectors for Word Representation](https://nlp.stanford.edu/projects/glove/) is an algorithm for learning vector representations, just like jointly training a `nn.Embedding`layer along with other sub-modules in a given task.

In [47]:
glove = torchtext.vocab.GloVe(name="6B", dim=50)

word = "desperate"
emb_word = glove[word]
k = 5

get_top_k(k, emb_word, None, glove.vectors, device, lookup_token=lambda idx: glove.itos[idx])
get_top_k(k, emb_word, None, glove.vectors, device, most_similar=False, lookup_token=lambda idx: glove.itos[idx])
get_top_k(k, emb_word, None, glove.vectors, device, distance="cosine", lookup_token=lambda idx: glove.itos[idx])
get_top_k(k, emb_word, None, glove.vectors, device, most_similar=False, distance="cosine", lookup_token=lambda idx: glove.itos[idx])

Top k similar (l2):  desperately, vain, wanting, trouble, weary, 

Top k disimilar (l2):  non-families, 202-383-7824, 20003, www.star, officership, 

Top k similar (cosine):  desperately, trouble, wanting, vain, weary, 

Top k disimilar (cosine):  landolt, preus, icct, yisheng, daojiong, 



In [52]:
for analogy in analogies_students:
    word_a, word_b, word_c = analogy
    emb_word_a = glove[word_a]
    emb_word_b = glove[word_b]
    emb_word_c = glove[word_c]

    new_embed = emb_word_b - emb_word_a + emb_word_c
    
    result = get_top_k(3, new_embed, None, glove.vectors, device, distance="cosine", return_res=True, lookup_token=lambda idx: glove.itos[idx])
    result = list(map(str.upper, result))
    
    if result:
        print(f"{word_a.upper()} is to {word_b.upper()} as {word_c.upper()} is to {result}")
    else:
        print("Some words in the analogy are not in the vocabulary.")

DOG is to FUR as CAT is to ['FURS', 'SILK', 'WOOL']
MAN is to LIVINGROOM as WOMAN is to ['PENPAL', 'SULTRINESS', 'NIGHTIE']
BIRD is to FEATHER as ELEPHANT is to ['PAW', 'FEATHERED', 'VELVETEEN']


We see that these embeddings perform much better, while also revealing some gender bias.

---

## Homework 🔬 (20 pts, teams of max. 1) **Forecasting with Seq2Seq** <a class="anchor" id="app-2"></a>

In this Homework, you'll train and test a Seq2Seq model (checkout the course for more info) to predict the next frames in a simplistic video forecasting scenario.

Tasks:
1. Train a simple Seq2Seq model for forecasting the next 10 frames, given the past 10 frames as context (`5pt`)
2. Train an attention-enhanced Seq2Seq model for the same task (`5pt`)
3. Compare the performance of the two models, using at least 2 metrics of your choice (`5pt`)
4. Visualize attention scores for different timesteps, and discuss the results (`5pt`)

---
No project report is required. Your submission will consist in:
1. `.zip` or a public Github containing the source code. You can include all your homework in a single Jupyter notebook, provided that you include any necessary dependencies.
2. A `5-minute` presentation in `.pdf` format, which will be presented in the last week of this course. The presentation should be made similar to [this Overleaf template.](https://www.overleaf.com/latex/templates/cnu-beamer/ftpfhcwstgpy), and should include:
   - A brief description of the task
   - Choice of noise values
   - Configuration of your training strategy with noisy mixtures
   - Results and discussion
---

We'll be using the [Seq2Seq](https://arxiv.org/pdf/1409.3215) architecture (also known as "Sutskever model") to predict the next frames from [Moving MNIST dataset](https://paperswithcode.com/dataset/moving-mnist). Seq2Seq models have been used for sequence-to-sequence translation (hence the name), mostly in LanguageA-to-LanguageB tasks. Check out [this tutorial notebook](https://github.com/bentrevett/pytorch-seq2seq/blob/main/1%20-%20Sequence%20to%20Sequence%20Learning%20with%20Neural%20Networks.ipynb) on german-to-english translation.

We'll adapt the Seq2Seq architecture to work on 2D data (frames) instead of tokens, by utilizing a $2D \rightarrow 1D$ **encoder** that will process all input frames independently, followed by a LSTM network to model the sequence of temporal features, and a final $1D \rightarrow 2D$ **decoder** that will translate the predicted 1D features to 2D future frames.

Check out [these nice visuals](https://lena-voita.github.io/nlp_course/seq2seq_and_attention.html) on how Seq2Seq works (including adding attention).

In [ ]:
from torchvision.datasets import MovingMNIST

split_ratio = 10

transform = lambda x: x / 255.0

first_frames = MovingMNIST(root="data/", split="train", download=True, split_ratio=split_ratio, transform=transform)
last_frames = MovingMNIST(root="data/", split="test", download=True, split_ratio=split_ratio, transform=transform)

We'll be using the first 10 frames as input, and our task is to predict the following 10 frames.

In [ ]:
bs_ = 16

print("Shape of first frames (known): ", first_frames[:bs_].shape)
print("Shape of last frames (unknown): ", last_frames[:bs_].shape)
print(f"Value range: [{first_frames[:bs_].min()}, {first_frames[:bs_].max()}]")

In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
import ipywidgets as widgets
from IPython.display import display
import numpy as np

i = np.random.randint(0, len(first_frames))
all_frames = np.concatenate((first_frames[i], last_frames[i]), axis=0)

def display_frame(frame_number):
    plt.imshow(all_frames[frame_number].squeeze(), cmap="gray")
    plt.axis('off')
    plt.show()

frame_slider = widgets.IntSlider(min=0, max=all_frames.shape[0] - 1, step=1, description='Frame')
widgets.interact(display_frame, frame_number=frame_slider)

### Split and Create train/test loaders

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset, Dataset, DataLoader

class CustomMMNIST(Dataset):
    def __init__(self, first_fs, last_fs):
        self.first_ = first_fs
        self.last_ = last_fs

        assert len(self.first_) == len(self.last_)
    
    def __len__(self):
        return len(self.first_)

    def __getitem__(self, idx):
        return self.first_[idx], self.last_[idx]

indexes = list(range(len(first_frames)))
train_idx, test_idx = train_test_split(indexes, test_size=0.2, random_state=42)  

first_train, first_test = Subset(first_frames, train_idx), Subset(first_frames, test_idx)
last_train, last_test = Subset(last_frames, train_idx), Subset(last_frames, test_idx)

train_mnist = CustomMMNIST(first_train, last_train)
test_mnist = CustomMMNIST(first_test, last_test)

train_loader = DataLoader(train_mnist, batch_size=32, shuffle=True)
test_loader = DataLoader(test_mnist, batch_size=32, shuffle=False)

In [ ]:
len(train_mnist), len(test_mnist)

### Construct Seq2Seq model

The usual architecture of Seq2Seq is depicted as follows:

<div style="text-align: center;">
  <img src="imgs/seq2seq.png" alt="description" width="1200"/>
  <figcaption>Seq2Seq architecture. <a href="https://blog.suriya.app/2016-12-31-practical-seq2seq/">Source</a></figcaption>
</div>

We have an RNN encoder that processes the input sequence, followed by another RNN decoder that receives the final states of the encoder and generates the output sequence, element-by-element, using the previously generated output as input.

Before diving into Seq2Seq, we need to define an image encoder that will transform each frame into a feature vector, to be passed to the RNN:

$$\texttt{Frame}_i \rightarrow \texttt{ImageEncoder}(\cdot) \rightarrow \texttt{Feature Vector}_i \rightarrow \texttt{Seq2Seq}(\cdot) \rightarrow \texttt{ImageDecoder}(\cdot) \rightarrow \texttt{Frame}_{i+1} $$

In [ ]:
import torch.nn as nn

class ImageEncoder(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(ImageEncoder, self).__init__()
    
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 8, 3, 1, 1),
            nn.ReLU(),
            nn.Conv2d(8, 16, 4, 4, 0),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, 1, 1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, 4, 4, 0),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, out_ch, 3, 1, 1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 4, 4, 0)
        )

    def forward(self, frames):
        """
        Frames has shape (B', C, H, W), where B' = B * n_frames represents the "extended batch size"
        """
        return self.net(frames)

We construct a simple Conv2d encoder, that succesively downsaples the input $64\times 64$ image into a 1D fature vector with `out_ch` elements. Note that the above architecture is exclusively built for $64\times 64$ images, with higher resolutions not being reduced to a 1D vector.

In [ ]:
import einops

x, y = train_mnist[:32]
bs, t, _, _, _ = x.shape

x_ = einops.rearrange(x, "d1 d2 d3 d4 d5 -> (d1 d2) d3 d4 d5") # we merge the first two dimensions to process all images at once
out_ = ImageEncoder(1, 128)(x_)
out = einops.rearrange(out_, "(b t) c 1 1 -> b t c", b=bs, t=t) # (b, t, c) is the data format our LSTM will expect

y_ = einops.rearrange(y, "b t c h w -> (b t) c h w")
out_y_ = ImageEncoder(1, 128)(y_)
out_y = einops.rearrange(out_y_, "(b t) c 1 1 -> b t c", b=bs, t=y.shape[1])

print("Original input: ", x.shape)
print("Rearranged input: ", x_.shape)
print("Original output: ", out_.shape)
print("Rearranged output: ", out.shape)

Further, we need to reconstruct frames from abstract features returned by the RNN. Therefore, we need an `ImageDecoder` *somewhat symmetric* to the `ImageEncoder`.

In [ ]:
class ImageDecoder(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(ImageDecoder, self).__init__()
        
        self.net = nn.Sequential(
            nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=4, stride=4, padding=0),
            nn.BatchNorm2d(in_ch // 2),
            nn.ReLU(),
            nn.Conv2d(in_ch // 2, in_ch // 2, 3, 1, 1),
            nn.BatchNorm2d(in_ch // 2),
            nn.ReLU(),
            nn.ConvTranspose2d(in_ch // 2, in_ch // 4, kernel_size=4, stride=4, padding=0),
            nn.BatchNorm2d(in_ch // 4),
            nn.ReLU(),
            nn.Conv2d(in_ch // 4, in_ch // 4, 3, 1, 1),
            nn.BatchNorm2d(in_ch // 4),
            nn.ReLU(),
            nn.ConvTranspose2d(in_ch // 4, in_ch // 8, kernel_size=4, stride=4, padding=0),
            nn.BatchNorm2d(in_ch // 8),
            nn.ReLU(),
            nn.Conv2d(in_ch // 8, in_ch // 8, 3, 1, 1),
            nn.BatchNorm2d(in_ch // 8),
            nn.ReLU(), 
            nn.Conv2d(in_ch // 8, out_ch, 1, 1, 0)
        )

    def forward(self, features):
        """
        features - tensor of shape (B', C, 1, 1). 
        """
        return self.net(features)

In [ ]:
decoded = ImageDecoder(128, 1)(einops.rearrange(out, "b t c -> (b t) c 1 1"))

print("Decoded frame shape: ", decoded.shape)

The temporal features produced by `ImageDecoder` will be fed to a Seq2Seq architecture, which will return a similar tensor for the predicted last frames. Let's design separately the encoder and decoder parts of seq2seq:

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_layers, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        
        self.rnn = nn.LSTM(input_dim, hidden_dim, n_layers, dropout=dropout, batch_first=True)

    def forward(self, vector_features):
        outputs, (hidden, cell) = self.rnn(vector_features)

        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, n_layers, dropout=0.1):
        super().__init__()
        self.output_dim = output_dim  # output_dim should be = hidden dim of image features from ImageEncoder
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        
        self.rnn = nn.LSTM(input_dim, hidden_dim, n_layers, dropout=dropout, batch_first=True)
        
        if hidden_dim != output_dim:
            self.fc_out = nn.Linear(hidden_dim, output_dim)
        else:
            self.fc_out = nn.Identity()
        self.dropout = nn.Dropout(dropout)

    def forward(self, in_token, hidden, cell):
        output, (hidden, cell) = self.rnn(in_token, (hidden, cell))
        prediction = self.fc_out(output)
        
        return prediction, hidden, cell

In [ ]:
h, c = Encoder(out.shape[-1], 128, 4)(out)
print("Encoder hidden/cell states: ", h.shape, c.shape) # (num_lstm_layers, batch_size, hidden_dim)

dec = Decoder(128, 128, 128, 4)
in_token = torch.zeros((bs, 1, out_y.shape[-1]))

# we'll need to iterate over the target sequence step-by-step, compute the current output, and fed it as input in the next step
for i in range(y.shape[1]):
    out_d, h, c = dec(in_token, h, c)
    print(f"step={i} Decoder output/hidden/cell states: ", out_d.shape, h.shape, c.shape)

Combine an `Encoder` and a `Decoder` instance into a `Seq2Seq` model:

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, input_dim, n_layers, dropout=0.1, learnable_start_token=True):        
        super().__init__()

        self.encoder = Encoder(input_dim, input_dim, n_layers, dropout=dropout)
        self.decoder = Decoder(input_dim, input_dim, input_dim, n_layers, dropout=dropout)

        # We need a START token. We can either set it to 0 or learn it.
        self.start_token = nn.Parameter(torch.zeros((1, 1, input_dim)), requires_grad=learnable_start_token)
    
    def forward(self, source, target, teacher_forcing_ratio):
        
        batch_size = target.shape[0]
        trg_length = target.shape[1]

        outputs = torch.zeros_like(target).to(source.device)

        hidden, cell = self.encoder(source)

        # Duplicate start token for the current batch
        input = einops.repeat(self.start_token, "1 1 x -> rep 1 x", rep=batch_size)
        # input = source[:, -1].unsqueeze(1)
        
        for t in range(trg_length):
            output, hidden, cell = self.decoder(input, hidden, cell)

            outputs[:, t] = output[:, 0]
            
            teacher_force = torch.rand(1) < teacher_forcing_ratio
            input = target[:, t].unsqueeze(1) if teacher_force else output
            
        return outputs

In the above, we have defined a `teacher_forcing_ratio`, which is a number in `[0, 1]` representing the probability that in the next step we'll feed as input the true target frame, otherwise we'll feed the previously generated frame. This is a helpful trick during training, otherwise the forward process might get stuck in unrealistic solutions if the previous timestep wasn't good enough, hence a slower optimization process.

**Obiously, during inferece `teacher_forcing_ratio = 0` since we don't practically have access to the future frames.**

In [ ]:
ss = Seq2Seq(128, 4)
print("Seq2Seq output: ", ss(out, out_y, 0.1).shape)

Finally, let's wrap everything up:

In [ ]:
class ForecastModel(nn.Module):
    def __init__(self, in_ch, inner_dim, n_lstm_layers, dropout):
        super().__init__()

        self.in_ch = in_ch
        self.inner_dim = inner_dim
        self.n_lstm_layers = n_lstm_layers
        self.drop = dropout

        self.im_enc = ImageEncoder(in_ch, inner_dim)
        self.im_dec = ImageDecoder(inner_dim, in_ch)

        self.seq2seq = Seq2Seq(inner_dim, n_lstm_layers, dropout)

    def forward(self, source, target, teacher_forcing_ratio):

        b = source.shape[0]
        ts, tt = source.shape[1], target.shape[1]

        # Encode input frames
        source_ = einops.rearrange(source, "b t c h w -> (b t) c h w")
        source_ = self.im_enc(source_)
        source_ = einops.rearrange(source_, "(b t) c 1 1 -> b t c", b=b, t=ts)

        # Encode target frames
        target_ = einops.rearrange(target, "b t c h w -> (b t) c h w")
        target_ = self.im_enc(target_)
        target_ = einops.rearrange(target_, "(b t) c 1 1 -> b t c", b=b, t=tt)

        # Apply Seq2Seq
        ss_pred = self.seq2seq(source_, target_, teacher_forcing_ratio)
        ss_pred_ = einops.rearrange(ss_pred, "b t c -> (b t) c 1 1")

        # Reconstruct next frames
        frame_pred = self.im_dec(ss_pred_)
        frame_pred = einops.rearrange(frame_pred, "(b t) c h w -> b t c h w", b=b, t=tt)
        
        return frame_pred

In [ ]:
out = ForecastModel(1, 32, 4, 0.1)(x, y, 0.1)

print("Output shape: ", out.shape)

### Train Seq2Seq model

In [ ]:
in_ch = 1
inner_dim = 128
n_lstm_layers = 2
dropout = 0.3
teacher_forcing_ratio = 0.5

epochs = 5
lr = 1e-3
folder_path = "models/mmnist/"
file_name = "model.pth"

model = ForecastModel(in_ch, inner_dim, n_lstm_layers, dropout)

Instead of standard $\ell_1$ or $\ell_2$ losses we can use the `BCEWithLogitsLoss` for each predicted pixel - this is possible **only because images are binary, i.e. targets are \{0, 1\}**.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = torch.nn.BCEWithLogitsLoss()

In [ ]:
import importlib
import torch_utils
importlib.reload(torch_utils)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_losses, test_losses = torch_utils.train_loop_forecast(
    model, 
    train_loader, 
    optimizer, 
    criterion, 
    epochs, 
    teacher_forcing_ratio,
    test_loader=test_loader, 
    device=device, 
    folder_path=folder_path, 
    file_name=file_name
)

In [ ]:
i = -5

pred = model(x[i].unsqueeze(0).to(device), y[i].unsqueeze(0).to(device), 0)
pred = nn.functional.sigmoid(pred)
pred = pred.squeeze(0).cpu().detach().numpy()

all_frames_pred = np.concatenate((x[i], pred), axis=0)
all_frames = np.concatenate((x[i], y[i]), axis=0)

plt.figure(figsize=(20, 4))
for i in range(all_frames.shape[0]):
    plt.subplot(1, all_frames.shape[0], i+1)
    plt.imshow(all_frames[i].squeeze(), cmap="gray")
    plt.axis("off")
plt.figure(figsize=(20, 4))
for i in range(all_frames_pred.shape[0]):
    plt.subplot(1, all_frames_pred.shape[0], i+1)
    plt.imshow(all_frames_pred[i].squeeze(), cmap="gray")
    plt.axis("off")

In [ ]:
def display_frame(frame_number):
    plt.subplot(1, 2, 1)
    plt.title("Predicted")
    plt.imshow(all_frames_pred[frame_number].squeeze(), cmap="gray")
    plt.axis('off')
    plt.subplot(1, 2, 2)
    plt.title("GT")
    plt.imshow(all_frames[frame_number].squeeze(), cmap="gray")
    plt.axis("off")
    plt.show()

frame_slider = widgets.IntSlider(min=0, max=all_frames_pred.shape[0] - 1, step=1, description='Frame')
widgets.interact(display_frame, frame_number=frame_slider)

Observations:
- As we go deeper into the temporal dimension, the information regarding class digit fades
- The dynamics remain well-predicted, meaning that the `Seq2Seq` model successfully retains & uses information from previous frames
- The `ImageEncoder` and `ImageDecoder` could share some information in order for the digit structure to remain intact after decoding

### Seq2Seq + Self-Attention

Let's add a `MultiheadAttention` attention module over the output temporal features:

<div style="text-align: center;">
  <img src="imgs/seq2seq_at.png" alt="description" width="1000"/>
  <figcaption>Seq2Seq + Self-Attention. <a href="https://blog.suriya.app/2016-12-31-practical-seq2seq/">Source</a></figcaption>
</div>

In [ ]:
import importlib
import torch_utils
importlib.reload(torch_utils)

class ForecastModelAtt(nn.Module):
    def __init__(self, in_ch, inner_dim, n_lstm_layers, dropout):
        super().__init__()

        self.in_ch = in_ch
        self.inner_dim = inner_dim
        self.n_lstm_layers = n_lstm_layers
        self.drop = dropout

        self.im_enc = ImageEncoder(in_ch, inner_dim)
        self.im_dec = ImageDecoder(inner_dim, in_ch)

        self.seq2seq = Seq2Seq(inner_dim, n_lstm_layers, dropout)

        # this actually doesn't replicate the input to each head, but rather divides the input into n_heads and applies attention separately on each part
        # check the documentation
        self.att = nn.MultiheadAttention(inner_dim, num_heads=4, batch_first=True, dropout=0.5)
        
    def forward(self, source, target, teacher_forcing_ratio):

        b = source.shape[0]
        ts, tt = source.shape[1], target.shape[1]

        # Encode input frames
        source_ = einops.rearrange(source, "b t c h w -> (b t) c h w")
        source_ = self.im_enc(source_)
        source_ = einops.rearrange(source_, "(b t) c 1 1 -> b t c", b=b, t=ts)

        # Encode target frames
        target_ = einops.rearrange(target, "b t c h w -> (b t) c h w")
        target_ = self.im_enc(target_)
        target_ = einops.rearrange(target_, "(b t) c 1 1 -> b t c", b=b, t=tt)

        # Apply Seq2Seq
        ss_pred = self.seq2seq(source_, target_, teacher_forcing_ratio)

        # Apply attention
        ss_pred, att_weights = self.att(ss_pred, ss_pred, ss_pred)
        
        ss_pred_ = einops.rearrange(ss_pred, "b t c -> (b t) c 1 1")

        # Reconstruct next frames
        frame_pred = self.im_dec(ss_pred_)
        frame_pred = einops.rearrange(frame_pred, "(b t) c h w -> b t c h w", b=b, t=tt)
        
        return frame_pred

In [ ]:
in_ch = 1
inner_dim = 128
n_lstm_layers = 2
dropout = 0.2
teacher_forcing_ratio = 0.5

epochs = 5
lr = 1e-3
folder_path = "models/mmnist_att/"
file_name = "model.pth"

model_att = ForecastModelAtt(in_ch, inner_dim, n_lstm_layers, dropout)

optimizer = torch.optim.Adam(model_att.parameters(), lr=lr)
criterion = torch.nn.BCEWithLogitsLoss()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_losses, test_losses = torch_utils.train_loop_forecast(
    model_att, train_loader, optimizer, criterion, epochs, teacher_forcing_ratio,
    test_loader=test_loader, device=device, folder_path=folder_path, file_name=file_name
)

In [ ]:
i = -5

pred = model(x[i].unsqueeze(0).to(device), y[i].unsqueeze(0).to(device), 0)
pred = nn.functional.sigmoid(pred)
pred = pred.squeeze(0).cpu().detach().numpy()

pred_att = model_att(x[i].unsqueeze(0).to(device), y[i].unsqueeze(0).to(device), 0)
pred_att = nn.functional.sigmoid(pred_att)
pred_att = pred_att.squeeze(0).cpu().detach().numpy()

all_frames_pred_att = np.concatenate((x[i], pred_att), axis=0)
all_frames_pred = np.concatenate((x[i], pred), axis=0)
all_frames = np.concatenate((x[i], y[i]), axis=0)


def display_frame(frame_number):
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 3, 1)
    plt.title("Predicted")
    plt.imshow(all_frames_pred[frame_number].squeeze(), cmap="gray")
    plt.axis('off')
    plt.subplot(1, 3, 2)
    plt.title("Predicted Self-Att")
    plt.imshow(all_frames_pred_att[frame_number].squeeze(), cmap="gray")
    plt.axis('off')
    plt.subplot(1, 3, 3)
    plt.title("GT")
    plt.imshow(all_frames[frame_number].squeeze(), cmap="gray")
    plt.axis("off")
    plt.show()

frame_slider = widgets.IntSlider(min=0, max=all_frames_pred.shape[0] - 1, step=1, description='Frame')
widgets.interact(display_frame, frame_number=frame_slider)